## импорты

In [3]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy.stats import randint, gaussian_kde
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

colors = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73',
          'red': '#D55E00', 'purple': '#CC79A7', 'gray': '#999999'}

plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## загрузка данных

In [6]:
nums = 'Номера закупок для питона.md'
file = 'Все_закупки_объединённые.xlsx'

with open(nums, encoding='utf-8') as f:
    raw = f.read()

cartel_nums = set(n.strip() for n in raw.split(',') if n.strip())

df = pd.read_excel(file, dtype={'Номер': str})
df['is_cartel'] = df['Номер'].isin(cartel_nums).astype(int)

n_cartel = df['is_cartel'].sum()
print(f'всего закупок: {len(df)}')
print(f'картельных закупок: {df['is_cartel'].sum()} ({df['is_cartel'].mean()*100:.1f}%)')
print(f'некартельных закупок: {len(df) - n_cartel}')

всего закупок: 15138
картельных закупок: 254 (1.7%)
некартельных закупок: 14884


## создание features

In [ ]:
# признаки для базовых моделей
df['несостоявшаяся'] = (df['Этап отбора'] == 'Признана несостоявшейся').astype(int)
df['процент_снижения'] = df['Процент снижения'].fillna(0)
df['снижение_низкое'] = (df['процент_снижения'] < 1.0).astype(int)
df['цена_договора'] = df['Цена договора'].fillna(df['НМЦ'])
df['цена_к_нмц'] = (df['цена_договора'] / df['НМЦ']).clip(0, 2)
df['log_нмц'] = np.log1p(df['НМЦ'])
reg_fr = df['Регион'].value_counts(normalize=True)
df['регион_частота'] = df['Регион'].map(reg_fr)
df['месяц'] = pd.to_datetime(df['Дата публикации'], errors='coerce').dt.month.fillna(0).astype(int)

# признаки для графов
reg_fail = df.groupby('Регион')['несостоявшаяся'].mean()
df['регион_доля_несостоявшихся'] = df['Регион'].map(reg_fail)

comp = df[df['ИНН победителя'].notna()]
reg_conc = (comp.groupby('Регион')['ИНН победителя']
    .apply(lambda x: x.value_counts(normalize=True).iloc[0] if len(x) > 0 else 0))
df['победитель_концентрация_в_регионе'] = df['Регион'].map(reg_conc).fillna(0)

inn_wins = df['ИНН победителя'].value_counts()
df['победитель_побед_всего'] = df['ИНН победителя'].map(inn_wins).fillna(0)

inn_reg_wins = df.groupby(['ИНН победителя', 'Регион']).size()
df['победитель_побед_в_регионе'] = df.apply(lambda row: inn_reg_wins.get((row['ИНН победителя'], row['Регион']), 0)
    if pd.notna(row['ИНН победителя']) else 0, axis=1)

inn_fail = df.groupby('ИНН победителя')['несостоявшаяся'].mean()
df['победитель_доля_несостоявшихся'] = df['ИНН победителя'].map(inn_fail).fillna(0)

features_base = ['log_нмц',
    'несостоявшаяся',
    'процент_снижения',
    'снижение_низкое',
    'цена_к_нмц',
    'регион_частота',
    'месяц']

features_graph = features_base + [
    'регион_доля_несостоявшихся',
    'победитель_концентрация_в_регионе',
    'победитель_побед_всего',
    'победитель_побед_в_регионе',
    'победитель_доля_несостоявшихся']

y = df['is_cartel']
df.head(5)

## train/test split + SMOTE

In [8]:
X_train_b, X_test_b, y_train, y_test = train_test_split(
    df[features_base], y, test_size=0.2, random_state=42, stratify=y)
X_train_g = df.loc[X_train_b.index, features_graph]
X_test_g = df.loc[X_test_b.index, features_graph]

smote = SMOTE(random_state=42)
X_train_b_res, y_train_res = smote.fit_resample(X_train_b, y_train)
X_train_g_res, _ = smote.fit_resample(X_train_g, y_train)

print(f'train: {len(X_train_b)} строк, картелей: {y_train.sum()}')
print(f'test: {len(X_test_b)} строк, картелей: {y_test.sum()}')
print(f'после SMOTE: {len(X_train_b_res)} строк, картелей: {y_train_res.sum()} ({y_train_res.mean()*100:.1f}%)')

train: 12110 строк, картелей: 203
test: 3028 строк, картелей: 51
после SMOTE: 23814 строк, картелей: 11907 (50.0%)


## модель 0: logistic regression

In [9]:
scaler = StandardScaler()
X_train_b_scaled = scaler.fit_transform(X_train_b)
X_test_b_scaled = scaler.transform(X_test_b)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_b_scaled, y_train)
y_pred_lr = lr.predict(X_test_b_scaled)

print(classification_report(y_test, y_pred_lr, target_names=['Норма', 'Картель']))
cm = confusion_matrix(y_test, y_pred_lr)
print(f'''Матрица: TN={cm[0,0]} FP={cm[0,1]} 
         FN={cm[1,0]}   TP={cm[1,1]}''')

              precision    recall  f1-score   support

       Норма       0.99      0.62      0.76      2977
     Картель       0.03      0.71      0.06        51

    accuracy                           0.62      3028
   macro avg       0.51      0.66      0.41      3028
weighted avg       0.98      0.62      0.75      3028

Матрица: TN=1849 FP=1128 
         FN=15   TP=36


## модель 1: random forest base

In [10]:
rf1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf1.fit(X_train_b_res, y_train_res)
y_pred_rf1 = rf1.predict(X_test_b)

print(classification_report(y_test, y_pred_rf1, target_names=['Норма', 'Картель']))
cm = confusion_matrix(y_test, y_pred_rf1)
print(f'''Матрица: TN={cm[0,0]} FP={cm[0,1]} 
         FN={cm[1,0]}   TP={cm[1,1]}''')

print('\nважность признаков:')
for feat, val in pd.Series(rf1.feature_importances_, index=features_base).sort_values(ascending=False).items():
    print(f'  {feat}: {val:.4f}')

              precision    recall  f1-score   support

       Норма       0.99      0.96      0.97      2977
     Картель       0.15      0.47      0.23        51

    accuracy                           0.95      3028
   macro avg       0.57      0.71      0.60      3028
weighted avg       0.98      0.95      0.96      3028

Матрица: TN=2845 FP=132 
         FN=27   TP=24

важность признаков:
  цена_к_нмц: 0.2467
  регион_частота: 0.2119
  log_нмц: 0.2007
  месяц: 0.1458
  процент_снижения: 0.1337
  несостоявшаяся: 0.0611
  снижение_низкое: 0.0000


## RandomizedSearchCV

In [15]:
param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None] + list(range(3, 31)),
    'min_samples_leaf': randint(1, 21)}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_distributions=param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=0)

random_search.fit(X_train_g_res, y_train_res)

print(f'лучшие параметры: {random_search.best_params_}')
print(f'F1 на кросс-валидации: {random_search.best_score_:.4f}')

лучшие параметры: {'max_depth': 29, 'min_samples_leaf': 1, 'n_estimators': 363}
F1 на кросс-валидации: 0.9746


## модель 2: random forest graph

In [16]:
rf2 = random_search.best_estimator_
y_pred_rf2 = rf2.predict(X_test_g)

print(classification_report(y_test, y_pred_rf2, target_names=['Норма', 'Картель']))
cm = confusion_matrix(y_test, y_pred_rf2)
print(f'''Матрица: TN={cm[0,0]} FP={cm[0,1]}
         FN={cm[1,0]}   TP={cm[1,1]}''')

print('\nважность признаков:')
for feat, val in pd.Series(rf2.feature_importances_, index=features_graph).sort_values(ascending=False).items():
    print(f'  {feat}: {val:.4f}')

              precision    recall  f1-score   support

       Норма       0.99      0.96      0.98      2977
     Картель       0.18      0.53      0.27        51

    accuracy                           0.95      3028
   macro avg       0.59      0.74      0.62      3028
weighted avg       0.98      0.95      0.96      3028

Матрица: TN=2855 FP=122
         FN=24   TP=27

важность признаков:
  цена_к_нмц: 0.2103
  log_нмц: 0.1484
  процент_снижения: 0.0949
  месяц: 0.0947
  победитель_концентрация_в_регионе: 0.0929
  регион_доля_несостоявшихся: 0.0806
  регион_частота: 0.0776
  победитель_побед_всего: 0.0737
  победитель_побед_в_регионе: 0.0554
  несостоявшаяся: 0.0429
  победитель_доля_несостоявшихся: 0.0286
  снижение_низкое: 0.0000


## модель 3: XGBoost base

In [17]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

print(f'scale_pos_weight = {scale:.1f}\n')

xgb1 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss',
    verbosity=0)

xgb1.fit(X_train_b, y_train)
y_pred_xgb1 = xgb1.predict(X_test_b)

print(classification_report(y_test, y_pred_xgb1, target_names=['Норма', 'Картель']))
cm = confusion_matrix(y_test, y_pred_xgb1)
print(f'''Матрица: TN={cm[0,0]} FP={cm[0,1]} 
         FN={cm[1,0]}   TP={cm[1,1]}''')

print('\nважность признаков:')
for feat, val in pd.Series(xgb1.feature_importances_, index=features_base).sort_values(ascending=False).items():
    print(f'  {feat}: {val:.4f}')

scale_pos_weight = 58.7

              precision    recall  f1-score   support

       Норма       0.99      0.93      0.96      2977
     Картель       0.11      0.49      0.18        51

    accuracy                           0.92      3028
   macro avg       0.55      0.71      0.57      3028
weighted avg       0.98      0.92      0.95      3028

Матрица: TN=2772 FP=205 
         FN=26   TP=25

важность признаков:
  процент_снижения: 0.3666
  регион_частота: 0.2051
  месяц: 0.1220
  цена_к_нмц: 0.1138
  log_нмц: 0.1078
  несостоявшаяся: 0.0847
  снижение_низкое: 0.0000


## модель 4: XGBoost graph

In [18]:
xgb2 = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss',
    verbosity=0)

xgb2.fit(X_train_g, y_train)
y_pred_xgb2 = xgb2.predict(X_test_g)

print(classification_report(y_test, y_pred_xgb2, target_names=['Норма', 'Картель']))
cm = confusion_matrix(y_test, y_pred_xgb2)
print(f'''Матрица: TN={cm[0,0]} FP={cm[0,1]} 
         FN={cm[1,0]}   TP={cm[1,1]}''')

print('\nважность признаков:')
for feat, val in pd.Series(xgb2.feature_importances_, index=features_graph).sort_values(ascending=False).items():
    print(f'  {feat}: {val:.4f}')

              precision    recall  f1-score   support

       Норма       0.99      0.92      0.96      2977
     Картель       0.12      0.63      0.21        51

    accuracy                           0.92      3028
   macro avg       0.56      0.78      0.58      3028
weighted avg       0.98      0.92      0.94      3028

Матрица: TN=2749 FP=228 
         FN=19   TP=32

важность признаков:
  процент_снижения: 0.2794
  регион_доля_несостоявшихся: 0.1492
  регион_частота: 0.1351
  победитель_побед_всего: 0.0863
  победитель_концентрация_в_регионе: 0.0625
  победитель_побед_в_регионе: 0.0587
  месяц: 0.0582
  цена_к_нмц: 0.0557
  log_нмц: 0.0489
  несостоявшаяся: 0.0368
  победитель_доля_несостоявшихся: 0.0291
  снижение_низкое: 0.0000


## сравнение всех моделей

In [19]:
results = [
    ('LogReg - базовые признаки', y_pred_lr),
    ('RF - базовые признаки', y_pred_rf1),
    ('RF - граф-признаки + RandomSearch', y_pred_rf2),
    ('XGBoost - базовые признаки', y_pred_xgb1),
    ('XGBoost - граф-признаки', y_pred_xgb2)]

print(f'{'Модель':<40} {'Precision':>10} {'Recall':>10} {'F1':>8} {'TP':>5} {'FP':>5}')
print('-' * 83)

for name, y_p in results:
    p = precision_score(y_test, y_p)
    r = recall_score(y_test, y_p)
    f = f1_score(y_test, y_p)
    cm = confusion_matrix(y_test, y_p)
    tp = cm[1, 1]
    fp = cm[0, 1]
    print(f'{name:<40} {p:>10.3f} {r:>10.3f} {f:>8.3f} {tp:>5} {fp:>5}')

Модель                                    Precision     Recall       F1    TP    FP
-----------------------------------------------------------------------------------
LogReg - базовые признаки                     0.031      0.706    0.059    36  1128
RF - базовые признаки                         0.154      0.471    0.232    24   132
RF - граф-признаки + RandomSearch             0.181      0.529    0.270    27   122
XGBoost - базовые признаки                    0.109      0.490    0.178    25   205
XGBoost - граф-признаки                       0.123      0.627    0.206    32   228


## графики

In [20]:
names = {
    'log_нмц': 'Логарифм НМЦ',
    'несостоявшаяся': 'Торги не состоялись',
    'процент_снижения': 'Процент снижения цены',
    'снижение_низкое': 'Снижение < 1%',
    'цена_к_нмц': 'Отношение цены к НМЦ',
    'регион_частота': 'Частота региона',
    'месяц': 'Месяц публикации',
    'регион_доля_несостоявшихся': 'Доля не состоявшихся в регионе',
    'победитель_концентрация_в_регионе': 'Концентрация в регионе',
    'победитель_побед_всего': 'Всего побед у победителя',
    'победитель_побед_в_регионе': 'Побед в регионе',
    'победитель_доля_несостоявшихся': 'Доля не состоявшихся у победителя'}

plot_features = [
    ('цена_к_нмц', 'Отношение цены договора к НМЦ', (0, 1.05)),
    ('процент_снижения', 'Процент снижения цены', (0, 0.6))]

fig, axes = plt.subplots(2, 1, figsize=(7.5, 8))

for ax, (feat, title, xlim) in zip(axes, plot_features):
    val0 = df.loc[df['is_cartel'] == 0, feat].dropna()
    val1 = df.loc[df['is_cartel'] == 1, feat].dropna()
    val0 = val0[(val0 >= xlim[0]) & (val0 <= xlim[1])]
    val1 = val1[(val1 >= xlim[0]) & (val1 <= xlim[1])]

    grid = np.linspace(xlim[0], xlim[1], 400)
    kde0 = gaussian_kde(val0, bw_method='scott')
    kde1 = gaussian_kde(val1, bw_method='scott')

    ax.fill_between(grid, kde0(grid), alpha=0.45, color=colors['blue'], label=f'Норма (n = {len(val0)})')
    ax.fill_between(grid, kde1(grid), alpha=0.45, color=colors['red'], label=f'Картель (n = {len(val1)})')
    ax.plot(grid, kde0(grid), color=colors['blue'], lw=1.5)
    ax.plot(grid, kde1(grid), color=colors['red'], lw=1.5)
    ax.axvline(val0.mean(), ls='--', color=colors['blue'], lw=1, alpha=0.7)
    ax.axvline(val1.mean(), ls='--', color=colors['red'], lw=1, alpha=0.7)

    ax.set_title(title)
    ax.set_xlabel('Значение признака')
    ax.set_ylabel('Плотность')
    ax.set_xlim(xlim)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig1_feature_distributions.png', dpi=300)
plt.close()

print('сохранён fig1_feature_distributions.png')

сохранён fig1_feature_distributions.png


## ROC-кривые

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))

roc_models = [
    ('LogReg', lr, X_test_b_scaled, colors['gray']),
    ('RF base', rf1, X_test_b, colors['purple']),
    ('RF graph', rf2, X_test_g, colors['green']),
    ('XGB base', xgb1, X_test_b, colors['orange']),
    ('XGB graph', xgb2, X_test_g, colors['red'])]

for name, model, X_test, color in roc_models:
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], '--', color='black', lw=1, alpha=0.5)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig('fig2_roc_curves.png', dpi=300)
plt.close()

print('сохранён fig2_roc_curves.png')

## precision-recall кривые

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))

pr_models = [
    ('LogReg', lr, X_test_b_scaled, colors['gray']),
    ('RF base', rf1, X_test_b, colors['purple']),
    ('RF graph', rf2, X_test_g, colors['green']),
    ('XGB base', xgb1, X_test_b, colors['orange']),
    ('XGB graph', xgb2, X_test_g, colors['red'])]

for name, model, X_test, color in pr_models:
    proba = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(rec, prec, label=f'{name} (AP = {ap:.3f})', color=color, lw=2)

base = y_test.mean()
ax.axhline(base, ls='--', color='black', lw=1, alpha=0.5, label=f'Случайная модель (AP = {base:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('fig3_pr_curves.png', dpi=300)
plt.close()

print('сохранён fig3_pr_curves.png')

## матрицы ошибок

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 7))

cm_models = [
    ('RF base', y_pred_rf1),
    ('RF graph', y_pred_rf2),
    ('XGB base', y_pred_xgb1),
    ('XGB graph', y_pred_xgb2)]

for ax, (name, y_p) in zip(axes.flat, cm_models):
    cm = confusion_matrix(y_test, y_p)
    ax.imshow(cm, cmap='Blues')

    for i in range(2):
        for j in range(2):
            ax.text(
                j, i, cm[i, j],
                ha='center',
                va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black',
                fontsize=12,
                fontweight='bold')

    ax.set_title(name)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Норма', 'Картель'])
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Норма', 'Картель'])
    ax.set_xlabel('Предсказание')
    ax.set_ylabel('Истинный класс')

plt.tight_layout()
plt.savefig('fig4_confusion_matrices.png', dpi=300)
plt.close()

print('сохранён fig4_confusion_matrices.png')

## важность признаков

In [27]:
fig, axes = plt.subplots(2, 1, figsize=(7.5, 9))

imp_rf = pd.Series(rf2.feature_importances_, index=features_graph).sort_values()
axes[0].barh([names.get(f, f) for f in imp_rf.index], imp_rf.values, color=colors['green'])
axes[0].set_title('Random Forest')
axes[0].set_xlabel('Важность')

imp_xgb = pd.Series(xgb2.feature_importances_, index=features_graph).sort_values()
axes[1].barh([names.get(f, f) for f in imp_xgb.index], imp_xgb.values, color=colors['blue'])
axes[1].set_title('XGBoost')
axes[1].set_xlabel('Важность')

plt.tight_layout()
plt.savefig('fig5_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

print('сохранён fig5_feature_importance.png')

сохранён fig5_feature_importance.png


## bootstrap CI + cross-validation

In [35]:
def bootstrap_ci(y_true, y_pred, n_boot=2000, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    f1_list = []
    prec_list = []
    rec_list = []

    for _ in range(n_boot):
        pos_s = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        neg_s = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([pos_s, neg_s])

        yt = y_true[idx]
        yp = y_pred[idx]

        f1_list.append(f1_score(yt, yp, zero_division=0))
        prec_list.append(precision_score(yt, yp, zero_division=0))
        rec_list.append(recall_score(yt, yp, zero_division=0))

    return {
        'f1': (np.percentile(f1_list, 2.5), np.percentile(f1_list, 97.5)),
        'prec': (np.percentile(prec_list, 2.5), np.percentile(prec_list, 97.5)),
        'rec': (np.percentile(rec_list, 2.5), np.percentile(rec_list, 97.5)),
        'f1_point': f1_score(y_true, y_pred, zero_division=0),
        'all_f1': np.array(f1_list)
    }


preds = {'LogReg': y_pred_lr, 'RF base': y_pred_rf1, 'RF graph': y_pred_rf2,
        'XGB base': y_pred_xgb1, 'XGB graph': y_pred_xgb2}

boot_results = {}

print(f"{'Модель':<12} {'F1':<8} {'95% CI F1':<18} {'Prec CI':<18} {'Rec CI':<18}")
print('-' * 78)

for name, y_pred in preds.items():
    res = bootstrap_ci(y_test, y_pred)
    boot_results[name] = res

    f1_text = f"[{res['f1'][0]:.3f}, {res['f1'][1]:.3f}]"
    prec_text = f"[{res['prec'][0]:.3f}, {res['prec'][1]:.3f}]"
    rec_text = f"[{res['rec'][0]:.3f}, {res['rec'][1]:.3f}]"

    print(f"{name:<12} {res['f1_point']:<8.3f} {f1_text:<18} {prec_text:<18} {rec_text:<18}")

Модель       F1       95% CI F1          Prec CI            Rec CI            
------------------------------------------------------------------------------
LogReg       0.059    [0.049, 0.070]     [0.025, 0.036]     [0.569, 0.824]    
RF base      0.232    [0.170, 0.295]     [0.112, 0.199]     [0.333, 0.608]    
RF graph     0.270    [0.203, 0.339]     [0.137, 0.231]     [0.392, 0.667]    
XGB base     0.178    [0.130, 0.227]     [0.079, 0.139]     [0.353, 0.627]    
XGB graph    0.206    [0.164, 0.249]     [0.098, 0.150]     [0.490, 0.765]    


## 5-кратная cross-validation

In [34]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))])

cv_models = [
    ('LogReg', lr_pipe, X_train_b, y_train),
    ('RF base', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42), X_train_b, y_train),
    ('RF graph', RandomForestClassifier(**random_search.best_params_, class_weight='balanced', random_state=42), X_train_g, y_train),
    ('XGB base', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, scale_pos_weight=scale, eval_metric='logloss', random_state=42, verbosity=0), X_train_b, y_train),
    ('XGB graph', XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, scale_pos_weight=scale, eval_metric='logloss', random_state=42, verbosity=0), X_train_g, y_train)]

cv_results = {}

print(f"{'Модель':<12} {'mean F1':>8} {'std':>8} {'min':>8} {'max':>8}")
print('-' * 50)

for name, model, X_train, y_train_cv in cv_models:
    scores = cross_val_score(model, X_train, y_train_cv, cv=skf, scoring='f1', n_jobs=-1)
    cv_results[name] = scores

    print(f'{name:<12} {scores.mean():>8.3f} {scores.std():>8.3f} {scores.min():>8.3f} {scores.max():>8.3f}')

Модель        mean F1      std      min      max
--------------------------------------------------
LogReg          0.066    0.005    0.058    0.071
RF base         0.152    0.091    0.042    0.320
RF graph        0.174    0.082    0.085    0.327
XGB base        0.212    0.027    0.171    0.246
XGB graph       0.229    0.024    0.197    0.252


## график доверительных интервалов

In [37]:
model_names = [name for name, *_ in cv_models]
plot_colors = [colors['gray'], colors['orange'], colors['blue'], colors['red'], colors['green']]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cv_data = [cv_results[name] for name in model_names]
bp = axes[0].boxplot(cv_data, patch_artist=True, notch=False, medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], plot_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

axes[0].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[0].set_ylabel('F1-score')
axes[0].set_title('5-кратная CV')
axes[0].set_ylim(bottom=0)

f1_points = [boot_results[name]['f1_point'] for name in model_names]
ci_lo = [boot_results[name]['f1'][0] for name in model_names]
ci_hi = [boot_results[name]['f1'][1] for name in model_names]

err_lo = [f - lo for f, lo in zip(f1_points, ci_lo)]
err_hi = [hi - f for f, hi in zip(f1_points, ci_hi)]

x = range(len(model_names))

for i, (xi, yi, color) in enumerate(zip(x, f1_points, plot_colors)):
    axes[1].errorbar(xi, yi, yerr=[[err_lo[i]], [err_hi[i]]], fmt='o', color=color, capsize=6, markersize=8, linewidth=2)

axes[1].set_xticks(list(x))
axes[1].set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
axes[1].set_ylabel('F1-score')
axes[1].set_title('Bootstrap 95% CI для F1')
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.savefig('fig6_ci_bootstrap_cv.png', dpi=300, bbox_inches='tight')
plt.close()

print('сохранён fig6_ci_bootstrap_cv.png')

сохранён fig6_ci_bootstrap_cv.png


## итоговая таблица с CI

In [45]:
print(f"{'Модель':<12} {'F1 тест':>8} {'95% CI F1':>20} {'CV mean':>10} {'CV std':>10}")
print('=' * 64)

for name in model_names:
    f1_test = boot_results[name]['f1_point']
    ci = boot_results[name]['f1']
    cv_mean = cv_results[name].mean()
    cv_std = cv_results[name].std()
    ci_text = f'[{ci[0]:.3f}, {ci[1]:.3f}]'

    print(f'{name:<12} {f1_test:>8.3f} {ci_text:>20} {cv_mean:>10.3f} {cv_std:>10.3f}')

Модель        F1 тест            95% CI F1    CV mean     CV std
LogReg          0.059       [0.049, 0.070]      0.066      0.005
RF base         0.232       [0.170, 0.295]      0.152      0.091
RF graph        0.270       [0.203, 0.339]      0.174      0.082
XGB base        0.178       [0.130, 0.227]      0.212      0.027
XGB graph       0.206       [0.164, 0.249]      0.229      0.024


## таблицы для работы

In [ ]:
def save_table(rows, col_labels, path, title, col_widths, fontsize=10):
    fig, ax = plt.subplots(figsize=(11, 0.55 * (len(rows) + 1.5)))
    ax.axis('off')
    ax.set_title(title, fontsize=11, pad=10, loc='left')

    table = ax.table(
        cellText=rows,
        colLabels=col_labels,
        cellLoc='center',
        colLoc='center',
        loc='center',
        colWidths=col_widths)

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.7)

    for j in range(len(col_labels)):
        cell = table[(0, j)]
        cell.set_facecolor(colors['blue'])
        cell.set_text_props(color='white', weight='bold')

    for i in range(1, len(rows) + 1):
        for j in range(len(col_labels)):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#F5F5F5')

    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()

model_titles = {
    'LogReg': 'Логистическая регрессия',
    'RF base': 'RF - базовые признаки',
    'RF graph': 'RF - граф-признаки',
    'XGB base': 'XGBoost - базовые признаки',
    'XGB graph': 'XGBoost - граф-признаки'}

table2_rows = []
for name in model_names:
    res = boot_results[name]
    table2_rows.append([
        model_titles[name],
        f"{res['f1_point']:.3f}",
        f"[{res['f1'][0]:.3f}; {res['f1'][1]:.3f}]",
        f"[{res['prec'][0]:.3f}; {res['prec'][1]:.3f}]",
        f"[{res['rec'][0]:.3f}; {res['rec'][1]:.3f}]"])

save_table(
    table2_rows,
    col_labels=['Модель', 'F1', '95% CI F1', '95% CI Precision', '95% CI Recall'],
    path='table2_bootstrap_ci.png',
    title='Таблица 2. Точечные оценки и 95% доверительные интервалы метрик на тестовой выборке',
    col_widths=[0.28, 0.10, 0.20, 0.21, 0.21])

table3_rows = []
for name in model_names:
    res = boot_results[name]
    table3_rows.append([
        model_titles[name],
        f"{res['f1_point']:.3f}",
        f"[{res['f1'][0]:.3f}; {res['f1'][1]:.3f}]",
        f'{cv_results[name].mean():.3f}',
        f'{cv_results[name].std():.3f}'])

save_table(
    table3_rows,
    col_labels=['Модель', 'F1 на тесте', '95% CI F1', 'CV mean F1', 'CV std'],
    path='table3_summary.png',
    title='Таблица 3. Сравнение bootstrap CI и 5-кратной кросс-валидации без SMOTE',
    col_widths=[0.28, 0.13, 0.24, 0.14, 0.10])

print('сохранён table2_bootstrap_ci.png')
print('сохранён table3_summary.png')